# План экспериментов: 10 гипотез → как проверить → ожидаемый эффект

Этот ноутбук — отдельный рабочий документ для команды.  
Его цель — превратить идеи по улучшению baseline в последовательный и воспроизводимый план экспериментов.

## Контекст задачи

Нужно предсказывать `target_2h` по временным точкам для каждого маршрута на основе:
- `route_id`
- `office_from_id`
- `timestamp`
- `status_1 ... status_8`

Целевая метрика соревнования:
- **WAPE + |Relative Bias|**

Следовательно, эксперименты должны:
1. уменьшать абсолютную ошибку,
2. контролировать систематическое завышение или занижение прогноза,
3. быть честно проверены на временной валидации.

## Как пользоваться этим ноутбуком

Для каждой гипотезы ниже зафиксированы:
- **идея / зачем**
- **что меняем**
- **как проверяем**
- **ожидаемый эффект**
- **риски / что может пойти не так**

Рекомендуемый порядок работы:
1. Сначала повторить baseline и сохранить его метрики.
2. Затем тестировать по **одной гипотезе за раз**.
3. После этого собрать лучшие изменения в ансамбль или в финальную модель.

---


## Шаблон фиксации результатов

Для каждого запуска желательно записывать:
- дата эксперимента,
- название гипотезы,
- признаки,
- модель,
- схема валидации,
- WAPE,
- Relative Bias,
- итоговая метрика,
- комментарий.

Ниже — заготовка таблицы для ведения экспериментов.


In [ ]:
import pandas as pd

experiment_log = pd.DataFrame(columns=[
    "exp_id",
    "hypothesis",
    "features",
    "model",
    "validation",
    "WAPE",
    "Relative_Bias",
    "Final_metric",
    "comment"
])

experiment_log


## Приоритет запуска

Если времени мало, начинать лучше в таком порядке:

1. **Лаги + rolling-признаки**
2. **CatBoost / LightGBM вместо Ridge**
3. **Календарные признаки**
4. **Калибровка под WAPE + |Relative Bias|**
5. **Ансамбль лучших моделей**

Это наиболее вероятный путь быстро превзойти baseline.


## Гипотеза 1. Лаги по статусам и target улучшают качество прогноза

**Зачем проверяем**  
Baseline почти не использует временную динамику. Для логистики текущий объем отгрузки зависит от того, что происходило в предыдущие 30–360 минут.

**Что меняем**
- Добавить лаги для `status_1 ... status_8` на 1, 2, 4, 6, 12 шагов.
- Добавить лаги для `target_2h` на 1, 2, 4, 6, 12 шагов.
- Сравнить варианты: только лаги статусов / лаги статусов + лаги target.

**Как проверяем**
- Оставить модель baseline, но заменить только набор признаков.
- Проверить на той же time-based валидации.
- Сравнить WAPE, Relative Bias и итоговую метрику.

**Ожидаемый эффект**  
Ожидается заметное снижение ошибки, потому что модель начнет учитывать краткосрочную инерцию процессов на маршруте.

**Риски**  
Лаги target могут дать утечку, если неверно строить признаки относительно момента прогноза.


## Гипотеза 2. Rolling-статистики лучше отражают накопленный поток

**Зачем проверяем**  
Одного значения признака в конкретный момент мало. Средние, максимумы и разбросы за последние несколько часов часто лучше описывают состояние маршрута.

**Что меняем**
- Добавить rolling mean / sum / max / std по `status_1 ... status_8`.
- Использовать окна 2, 4, 8, 12 шагов.
- Добавить rolling-агрегаты для `target_2h`, если это допустимо без утечки.

**Как проверяем**
- Сравнить baseline и baseline + rolling features.
- Отдельно проверить маленькие и большие окна.
- Оценить, не ухудшается ли bias.

**Ожидаемый эффект**  
Ожидается более стабильный прогноз и снижение шума, особенно на маршрутах с рваной динамикой.

**Риски**  
Слишком много rolling-признаков может переусложнить модель и добавить мультиколлинеарность.


## Гипотеза 3. Признаки воронки по статусам описывают процесс лучше, чем сырые статусы

**Зачем проверяем**  
Статусы выглядят как этапы обработки. Для прогноза полезны не только абсолютные значения, но и структура движения по этапам.

**Что меняем**
- Добавить сумму по всем статусам.
- Добавить доли каждого статуса в общей сумме.
- Добавить разности и отношения между соседними статусами.
- Добавить признаки типа 'ранние этапы / поздние этапы'.

**Как проверяем**
- Собрать отдельный набор funnel-features.
- Проверить их отдельно и вместе с лагами.
- Сравнить вклад признаков по feature importance.

**Ожидаемый эффект**  
Ожидается рост качества за счет лучшего понимания того, где именно находится товарный поток перед отгрузкой.

**Риски**  
При малых значениях статусов отношения могут быть нестабильными; нужны защиты от деления на ноль.


## Гипотеза 4. Календарные признаки улучшают прогноз за счет сезонности

**Зачем проверяем**  
В логистике сильны внутрисуточные и недельные циклы. Один и тот же маршрут в разные часы и дни недели может вести себя по-разному.

**Что меняем**
- Извлечь из `timestamp`: час, день недели, день месяца, номер 30-минутного слота.
- Добавить признак выходного/буднего дня.
- Добавить cyclical encoding: sin/cos для часа и дня недели.

**Как проверяем**
- Сравнить baseline без календаря и baseline + calendar features.
- Отдельно посмотреть, где прирост больше: будни, выходные, пиковые часы.

**Ожидаемый эффект**  
Ожидается умеренный, но устойчивый прирост за счет учета регулярной сезонности.

**Риски**  
Если временной диапазон обучения короткий, часть календарных закономерностей может быть плохо оценена.


## Гипотеза 5. CatBoost / LightGBM лучше линейной Ridge-модели

**Зачем проверяем**  
Ridge удобна как baseline, но плохо ловит нелинейные зависимости и взаимодействия признаков.

**Что меняем**
- Обучить CatBoostRegressor и LightGBMRegressor.
- Использовать те же признаки, что у baseline, затем расширенный набор с лагами и rolling.
- Сравнить single-output и multi-horizon стратегии.

**Как проверяем**
- Одинаковая временная валидация для всех моделей.
- Одинаковый набор признаков в честном сравнении.
- Сравнить не только итоговую метрику, но и стабильность по фолдам.

**Ожидаемый эффект**  
Ожидается один из самых сильных приростов относительно baseline.

**Риски**  
Бустинги чувствительны к качеству валидации и могут переобучаться на шумных маршрутах.


## Гипотеза 6. Отдельная модель на каждый горизонт точнее общей multi-output модели

**Зачем проверяем**  
Ближние и дальние горизонты прогноза устроены по-разному. Одна модель на все шаги может сглаживать различия между ними.

**Что меняем**
- Обучить отдельную модель для каждого горизонта прогноза.
- Сравнить с multi-output baseline.
- Попробовать разные наборы признаков на ближних и дальних горизонтах.

**Как проверяем**
- Посчитать метрику отдельно по каждому горизонту.
- Сравнить агрегированный результат по всей задаче.
- Проверить, не растет ли время инференса слишком сильно.

**Ожидаемый эффект**  
Ожидается прирост прежде всего на дальних шагах прогноза.

**Риски**  
Станет больше моделей и сложнее сопровождение пайплайна.


## Гипотеза 7. Агрегаты по маршруту и складу улучшают обобщение

**Зачем проверяем**  
В данных есть и маршрут, и склад. Для некоторых маршрутов полезно знать типичное поведение на уровне склада и средний профиль самого маршрута.

**Что меняем**
- Добавить средние/медианы/квантили target по `route_id`.
- Добавить средние/медианы/квантили target по `office_from_id`.
- Добавить rolling-агрегаты по складу.
- Добавить признаки отклонения текущих статусов от среднего профиля маршрута.

**Как проверяем**
- Сравнить модель без иерархических агрегатов и с ними.
- Отдельно посмотреть эффект на редких и на частых маршрутах.

**Ожидаемый эффект**  
Ожидается улучшение устойчивости модели и качества на маршрутах с нестабильной историей.

**Риски**  
Нужно аккуратно считать агрегаты только по прошлому, иначе возможна утечка.


## Гипотеза 8. Калибровка прогноза под WAPE + |Relative Bias| снижает итоговую метрику

**Зачем проверяем**  
Даже хорошая модель может систематически недооценивать или переоценивать общий объем. А bias прямо входит в соревновательную метрику.

**Что меняем**
- После обучения подобрать глобальный коэффициент масштабирования `k` на валидации.
- Проверить отдельный `k` по горизонту, складу или времени суток.
- Сравнить с некалиброванными предсказаниями.

**Как проверяем**
- Минимизировать на валидации именно `WAPE + |Relative Bias|`.
- Смотреть не только общую метрику, но и суммарный прогноз против суммарного факта.

**Ожидаемый эффект**  
Ожидается быстрый и практичный прирост без полной смены модели.

**Риски**  
Слишком агрессивная калибровка может чуть улучшить bias, но испортить WAPE.


## Гипотеза 9. Rolling-origin валидация даст более надежный выбор модели, чем один split

**Зачем проверяем**  
Один временной split может дать случайно оптимистичную или пессимистичную оценку. Для временных рядов лучше проверять несколько последовательных окон.

**Что меняем**
- Построить 3–5 rolling-origin фолдов.
- Для каждого фолда считать WAPE, Relative Bias и итоговую метрику.
- Выбирать модель по среднему и стабильности.

**Как проверяем**
- Сравнить ranking моделей на single split и rolling CV.
- Посмотреть дисперсию результата по фолдам.

**Ожидаемый эффект**  
Ожидается не прямой прирост качества, а снижение риска выбрать модель, которая не перенесется на private leaderboard.

**Риски**  
Увеличится время обучения и объем вычислений.


## Гипотеза 10. Ансамбль из нескольких моделей будет сильнее любой одной модели

**Зачем проверяем**  
Линейные и нелинейные модели ловят разные типы зависимостей. Усреднение часто улучшает итоговую стабильность.

**Что меняем**
- Собрать ансамбль из Ridge, CatBoost и LightGBM.
- Проверить простое среднее и взвешенное blending.
- Подобрать веса на валидации под итоговую метрику.

**Как проверяем**
- Сначала сравнить одиночные модели.
- Потом сравнить ансамбль с лучшей одиночной моделью.
- Проверить, не ухудшается ли bias.

**Ожидаемый эффект**  
Ожидается устойчивый финальный прирост, особенно если модели делают разные ошибки.

**Риски**  
Ансамбль усложняет прод и интерпретацию решения.


## Рекомендуемая последовательность экспериментов

### Этап 1. Быстрые и дешевые улучшения
1. Лаги  
2. Rolling-признаки  
3. Календарные признаки  
4. Funnel-features  
5. Калибровка под метрику

### Этап 2. Смена класса модели
6. CatBoost  
7. LightGBM  
8. Отдельные модели по горизонтам

### Этап 3. Финальная сборка
9. Иерархические агрегаты по маршруту и складу  
10. Ансамбль лучших решений

Такой порядок позволяет сначала дешево улучшить baseline, затем перейти к более сильным моделям, а в конце собрать финальную конфигурацию.


In [ ]:
# Пример минимального списка экспериментов на запуск
experiment_queue = [
    {"exp_id": "E01", "hypothesis": "Lags baseline", "status": "planned"},
    {"exp_id": "E02", "hypothesis": "Rolling features", "status": "planned"},
    {"exp_id": "E03", "hypothesis": "Calendar features", "status": "planned"},
    {"exp_id": "E04", "hypothesis": "CatBoost on expanded features", "status": "planned"},
    {"exp_id": "E05", "hypothesis": "Calibration for metric", "status": "planned"},
]
pd.DataFrame(experiment_queue)


## Итог

Этот план можно использовать как:
- основу для командного research-процесса,
- раздел `Experiment plan` в README,
- приложение к презентации на защите.

Главная идея: **не просто перебирать модели, а двигаться по гипотезам, каждая из которых объясняет, почему качество должно вырасти именно в этой задаче.**
